# Getting Started with EquiTile

This tutorial introduces the basics of EquiTile, a tile-based local learning framework.

## What You'll Learn

- Creating your first EquiTile model
- Training on a simple dataset
- Evaluating performance
- Using the builder pattern

## Prerequisites

```bash
pip install torch torchvision
```

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Import EquiTile
from bioplausible.models.equitile import EquiTile, create_production_config

## 1. Creating Your First Model

### Basic Construction

The simplest way to create an EquiTile model:

In [ ]:
model = EquiTile(
    neurons_per_tile=64,
    num_layers=4,
    tiles_per_layer=4,
    input_dim=784,  # MNIST flattened
    output_dim=10,  # 10 classes
    mode='pc',  # Predictive Coding mode (recommended)
)

print(f"Model created with {len(model.graph.tiles)} tiles")
print(f"Number of edges: {len(model.graph.edges)}")

### Using Configuration Factories

For common use cases, use the configuration factories:

In [ ]:
# Production configuration (recommended for deployment)
config = create_production_config(
    neurons_per_tile=64,
    num_layers=4,
    tiles_per_layer=4,
)

model = EquiTile(
    neurons_per_tile=config.neurons_per_tile,
    num_layers=config.num_layers,
    tiles_per_layer=config.tiles_per_layer,
    input_dim=784,
    output_dim=10,
    mode=config.mode,
)

print(f"Production model: mode={config.mode}")

### Using the Builder Pattern (Recommended)

The builder pattern provides a fluent API:

In [ ]:
from bioplausible.models.equitile.builder import EquiTileBuilder

# Production-ready model
model = (EquiTileBuilder.production(input_dim=784, output_dim=10)
    .with_learning_rate(0.01)
    .with_dropout(0.1)
    .build())

print(f"Built model with builder")

## 2. Preparing Data

Let's create a simple synthetic dataset for demonstration:

In [ ]:
# Create synthetic classification data
n_samples = 1000
input_dim = 784
n_classes = 10

X = torch.randn(n_samples, input_dim)
y = torch.randint(0, n_classes, (n_samples,))

# Create DataLoader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset: {n_samples} samples, {n_classes} classes")
print(f"Batch size: 32")

## 3. Training Loop

### Basic Training

In [ ]:
# Simple training loop
n_epochs = 5

for epoch in range(n_epochs):
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    
    for X_batch, y_batch in dataloader:
        stats = model.train_step(X_batch, y_batch)
        total_loss += stats['loss']
        total_acc += stats['accuracy']
        n_batches += 1
    
    avg_loss = total_loss / n_batches
    avg_acc = total_acc / n_batches
    
    print(f"Epoch {epoch+1}/{n_epochs}: loss={avg_loss:.4f}, accuracy={avg_acc:.4f}")

### Using Training Context

The `TrainingContext` provides automatic logging and checkpointing:

In [ ]:
from bioplausible.models.equitile.builder import TrainingContext
import tempfile

# Create a temporary checkpoint path
with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    checkpoint_path = f.name

with TrainingContext(model, log_interval=10, checkpoint_path=checkpoint_path) as ctx:
    for epoch in range(3):  # Fewer epochs for demo
        for X_batch, y_batch in dataloader:
            stats = ctx.train_step(X_batch, y_batch)
        
        if ctx.should_checkpoint(epoch):
            ctx.save_checkpoint(epoch)
            print(f"  Checkpoint saved at epoch {epoch}")

## 4. Inference

### Basic Inference

In [ ]:
# Set model to eval mode
model.eval()

# Run inference on a batch
with torch.no_grad():
    X_test = torch.randn(10, 784)
    logits = model(X_test)
    predictions = logits.argmax(dim=-1)
    
print(f"Predictions: {predictions.tolist()}")

### Using Inference Context

In [ ]:
from bioplausible.models.equitile.builder import InferenceContext

with InferenceContext(model) as ctx:
    # Get logits
    logits = ctx.predict(X_test)
    
    # Get probabilities
    probs = ctx.predict_proba(X_test)
    
    # Get class predictions
    classes = ctx.predict_class(X_test)
    
print(f"Classes: {classes.tolist()}")
print(f"Top probability: {probs.max(dim=-1).values.mean().item():.4f}")

## 5. Model Inspection

EquiTile provides detailed model statistics:

In [ ]:
# Get model statistics
stats = model.get_stats()

print("Model Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 6. Profiling

Profile your model's performance:

In [ ]:
from bioplausible.models.equitile import EquiTileProfiler

profiler = EquiTileProfiler(model)

# Profile a training step
with profiler.profile(batch_size=32):
    model.train_step(X_batch, y_batch)

# Print report
profiler.print_report()

## 7. Saving and Loading

### Save Checkpoint

In [ ]:
# Save checkpoint
model.save_checkpoint('equitile_checkpoint.pt', metadata={
    'epoch': n_epochs,
    'config': 'production',
})
print("Checkpoint saved")

### Load Checkpoint

In [ ]:
# Create new model with same architecture
new_model = EquiTileBuilder.production(input_dim=784, output_dim=10).build()

# Load checkpoint
metadata = new_model.load_checkpoint('equitile_checkpoint.pt')
print(f"Loaded checkpoint: {metadata}")

## Summary

In this tutorial, you learned:

1. **Creating models**: Direct construction, config factories, and builder pattern
2. **Training**: Basic loops and TrainingContext
3. **Inference**: Direct calls and InferenceContext
4. **Inspection**: Model statistics
5. **Profiling**: Performance analysis
6. **Persistence**: Saving and loading checkpoints

## Next Steps

- Try the **Builder Pattern** tutorial for advanced model construction
- Explore **Multi-GPU Training** for scaling
- Check out **Research Experiments** for experiment tracking

## Exercises

1. Modify the architecture (neurons_per_tile, tiles_per_layer) and observe performance changes
2. Try EP mode instead of PC mode
3. Add dropout and weight decay to improve generalization
4. Profile different batch sizes to find optimal performance